# R&D-Weighted Structural Gap

Rescales the raw square-footage structural gap by each MSA's advanced-industry employment location quotient (LQ), so R&D-intensive metros count more heavily than large-but-generic industrial markets. Also identifies and profiles the smallest-base R&D markets by growth rate.

**Requires:** run `01_main_model.ipynb` first.

In [ ]:
"""
R&D-WEIGHTED (ADVANCED-INDUSTRY-WEIGHTED) STRUCTURAL GAP BY MSA -- STANDALONE (v14b)
================================================================================
Builds a single, presentation-ready CSV with the advanced-industry-weighted
structural (space-market) gap for every MSA from the v14b model.

WHAT "R&D-WEIGHTED" MEANS HERE: the v14b script's Adv_Weighted_Gap_SF is the
raw Structural_Gap_SF multiplied by Adv_Weight_LQ, a per-MSA weight anchored
to each MSA's 2015-2018 average advanced-industry employment location
quotient (LQ_AdvInd_Emp), rescaled via LQ/(1+LQ) so it's bounded in (0,1).
MSAs with a heavier concentration of advanced-industry (R&D-adjacent)
employment get a weight closer to 1 (their space-market gap counts closer
to full value); MSAs with little advanced-industry presence get a weight
closer to 0 (their raw square-footage gap is discounted, since it's less
relevant to the R&D real estate story even if the raw SF number is large).
This is NOT the same as RD_Intensity or LQ_RD_Expenditure from the other
model in this project -- it is specific to advanced-industry EMPLOYMENT
concentration, applied here to the space/availability counterfactual.

REQUIRES: AvailSFTotal_AdvWeighted_Actual_vs_Counterfactual_ByMSA.csv
already exists in the working directory (exported by the main v14b script's
Section 8B/8C). Adv_Weighted_Gap_SF is not stored directly in that file, but
is fully recoverable as:
    Adv_Weighted_Gap_SF = Adv_Weighted_Available_SF_Total
                          - Adv_Weighted_Counterfactual_SF
since Adv_Weighted_Gap_SF = Structural_Gap_SF * Adv_Weight_LQ
                          = (Available_SF_Total - Counterfactual_Space_SF) * Adv_Weight_LQ
                          = Adv_Weighted_Available_SF_Total - Adv_Weighted_Counterfactual_SF
by construction (see the main script's Section 8B). This script does NOT
reload the raw panel or refit the model -- purely a reshape of already-
computed results, so it runs standalone off disk.

(Full original version history retained in git log / docs/methodology.md.)
"""

import pandas as pd
import numpy as np
import os

WEIGHTED_FILE   = "AvailSFTotal_AdvWeighted_Actual_vs_Counterfactual_ByMSA.csv"
UNWEIGHTED_FILE = "AvailSFTotal_Counterfactual_Results.csv"   # optional, for Market_Category only
OUTPUT_FILE     = "AvailSFTotal_AdvWeighted_Structural_Gap_By_MSA.csv"

if not os.path.exists(WEIGHTED_FILE):
    raise SystemExit(
        f"STOPPING -- '{WEIGHTED_FILE}' not found in the working directory. "
        f"Run the main v14b model script first (it exports this file in "
        f"Section 8B/8C) before running this script."
    )

df = pd.read_csv(WEIGHTED_FILE)

required_cols = {"MSA_Name", "Year", "Adv_Weighted_Available_SF_Total",
                  "Adv_Weighted_Counterfactual_SF", "Adv_Weight_LQ"}
missing = required_cols - set(df.columns)
if missing:
    raise SystemExit(
        f"STOPPING -- '{WEIGHTED_FILE}' is missing expected column(s): {missing}. "
        f"Was this file exported by the v14b script's Section 8B/8C, or an older version?"
    )

print(f"Loaded {WEIGHTED_FILE}: {len(df)} rows | "
      f"{df['MSA_Name'].nunique()} MSAs | Years {sorted(df['Year'].unique())}")

# ── Recover Adv_Weighted_Gap_SF (see docstring for the algebra) ──────
df["Adv_Weighted_Gap_SF"] = (
    df["Adv_Weighted_Available_SF_Total"] - df["Adv_Weighted_Counterfactual_SF"])

# ── Wide pivot: one row per MSA, one column per year ────────────────
gap_wide = df.pivot_table(index="MSA_Name", columns="Year",
                           values="Adv_Weighted_Gap_SF", aggfunc="mean")
gap_wide.columns = [f"AdvWeightedGap_{int(y)}" for y in gap_wide.columns]
gap_wide = gap_wide.reset_index()

# ── Per-MSA weight (constant across years -- anchored 2015-2018) ────
weight_by_msa = (df.groupby("MSA_Name")["Adv_Weight_LQ"]
                  .mean()  # constant per MSA already; mean is just a safe collapse
                  .rename("Adv_Weight_LQ_2015_2018_Anchor"))

# ── Summary columns ──────────────────────────────────────────────────
precovid_years = [y for y in df["Year"].unique() if y <= 2019]
covid_years    = [2020, 2021, 2022, 2023]

avg_precovid = (df[df["Year"].isin(precovid_years)]
                .groupby("MSA_Name")["Adv_Weighted_Gap_SF"].mean()
                .rename("Avg_AdvWeightedGap_PreCOVID_2006_2019"))
avg_covid = (df[df["Year"].isin(covid_years)]
             .groupby("MSA_Name")["Adv_Weighted_Gap_SF"].mean()
             .rename("Avg_AdvWeightedGap_COVID_2020_2023"))

latest_year = df["Year"].max()
latest_gap = (df[df["Year"] == latest_year]
              .set_index("MSA_Name")["Adv_Weighted_Gap_SF"]
              .rename(f"Latest_Year_AdvWeightedGap_{int(latest_year)}"))

out = (gap_wide
       .merge(weight_by_msa, on="MSA_Name", how="left")
       .merge(avg_precovid, on="MSA_Name", how="left")
       .merge(avg_covid, on="MSA_Name", how="left")
       .merge(latest_gap, on="MSA_Name", how="left"))

# ── Optional: Market_Category from the unweighted results file ──────
if os.path.exists(UNWEIGHTED_FILE):
    unw = pd.read_csv(UNWEIGHTED_FILE)
    if {"MSA_Name", "Year", "Market_Category"}.issubset(unw.columns):
        cat_mode = (unw[unw["Year"].isin(covid_years)]
                    .groupby("MSA_Name")["Market_Category"]
                    .agg(lambda s: s.mode().iat[0] if not s.mode().empty else np.nan)
                    .rename("Market_Category_Mode_2020_2023"))
        out = out.merge(cat_mode, on="MSA_Name", how="left")
        print(f"Merged Market_Category from {UNWEIGHTED_FILE}.")
    else:
        print(f"NOTE: {UNWEIGHTED_FILE} found but missing expected columns -- "
              f"Market_Category not merged.")
else:
    print(f"NOTE: {UNWEIGHTED_FILE} not found -- Market_Category column omitted "
          f"(this is optional context, not required).")

out = out.sort_values("Avg_AdvWeightedGap_COVID_2020_2023", ascending=False).reset_index(drop=True)
out.to_csv(OUTPUT_FILE, index=False)

print(f"\nSaved: {OUTPUT_FILE}")
print(f"  {len(out)} MSAs, {len(out.columns)} columns")
print(f"\nTop 10 by 2020-2023 avg R&D-weighted structural gap (surplus):")
show_cols = ["MSA_Name", "Adv_Weight_LQ_2015_2018_Anchor", "Avg_AdvWeightedGap_COVID_2020_2023"]
if "Market_Category_Mode_2020_2023" in out.columns:
    show_cols.append("Market_Category_Mode_2020_2023")
print(out[show_cols].head(10).to_string(index=False))
print(f"\nBottom 10 by 2020-2023 avg R&D-weighted structural gap (deficit):")
print(out[show_cols].tail(10).to_string(index=False))

In [ ]:
"""PART OF FINAL MODEL — v14b-consistent (100-MSA panel) [R&D-WEIGHTED]
Actual vs. Counterfactual — Top 20 Small-Base Markets (2021-2023)
===================================================================
Reads AvailSFTotal_AdvWeighted_Actual_vs_Counterfactual_ByMSA.csv (the
Section 8C R&D-weighted export from the v14b -- 100-MSA-trained --
dynamic-panel model) and AvailSFTotal_Counterfactual_Results.csv (for
Market_Category context only). Filters to the 20 MSAs identified as
smallest-by-recent-R&D-Weighted-base in the standalone small-base
script's Section 3B output, and plots each MSA's R&D-Weighted actual
vs. R&D-Weighted counterfactual Available SF Total, averaged over
2021-2023.

UPDATED FOR v14b (100-MSA-trained panel): unlike the v11->v12->v13
transitions, which only changed feature-set/retraining details without
touching which MSAs the model was trained on, v14b actually excludes
Hattiesburg, MS and Gulfport-Biloxi, MS from the ENTIRE pipeline (see
the main model's

(Full original version history retained in git log / docs/methodology.md.)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams.update({"font.family": "serif", "font.size": 11})

BLUE = "#1a3a5c"        # actual series -- matches every other chart in this project
GRAY = "#c9c9c9"        # counterfactual series -- lightened for stronger value contrast
GRAY_EDGE = "#999999"   # thin border on counterfactual bars so they stay defined
LABEL_COLOR = "#222222"  # gap-value text labels -- darkened + bolded for readability

INPUT_CSV_WEIGHTED = "AvailSFTotal_AdvWeighted_Actual_vs_Counterfactual_ByMSA.csv"
INPUT_CSV_RESULTS = "AvailSFTotal_Counterfactual_Results.csv"  # for Market_Category context only
OUTPUT_PNG = "smallbase_recent2021_2023_actual_vs_counterfactual_v14b.png"
OUTPUT_CSV = "SmallBase_Recent_Top20_Actual_vs_Counterfactual_v14b.csv"

RECENT_YEARS = [2021, 2022, 2023]

# The 20 MSAs -- still the list derived under the pre-v14b model. See
# the "THE 20-MSA LIST ITSELF" note above before trusting this
# unchanged under v14b.
TOP20_MSAS = [
    "McAllen-Edinburg-Mission, TX",
    "Urban Honolulu, HI",
    "Palm Bay-Melbourne-Titusville, FL",
    "Deltona-Daytona Beach-Ormond Beach, FL",
    "Baton Rouge, LA",
    "Fresno, CA",
    "Cape Coral-Fort Myers, FL",
    "Albuquerque, NM",
    "North Port-Bradenton-Sarasota, FL",
    "Spokane-Spokane Valley, WA",
    "Knoxville, TN",
    "Chattanooga, TN-GA",
    "Albany-Schenectady-Troy, NY",
    "Ogden, UT",
    "Kiryas Joel-Poughkeepsie-Newburgh, NY",
    "Colorado Springs, CO",
    "Little Rock-North Little Rock-Conway, AR",
    "New Orleans-Metairie, LA",
    "Omaha, NE-IA",
    "Winston-Salem, NC",
]

# ══════════════════════════════════════════════════════════════════
# 0. SANITY CHECK — confirm the R&D-weighted columns are present AND
#    that this is a 100-MSA (v14b) export, not a 102-MSA (v14 or
#    earlier) one.
# ══════════════════════════════════════════════════════════════════
_check = pd.read_csv(INPUT_CSV_WEIGHTED)
_required = {'MSA_Name', 'Year', 'Available_SF_Total', 'Counterfactual_Space_SF',
             'Adv_Weight_LQ', 'Adv_Weighted_Available_SF_Total', 'Adv_Weighted_Counterfactual_SF'}
_missing = _required - set(_check.columns)
if _missing:
    raise SystemExit(
        f"STOPPING -- {INPUT_CSV_WEIGHTED} is missing columns this script expects: {_missing}. "
        f"Re-run the main model script (v14b) to regenerate it before using this script."
    )
_n_msas = _check['MSA_Name'].nunique()
if _n_msas > 100:
    print(f"WARNING -- {INPUT_CSV_WEIGHTED} contains {_n_msas} MSAs, not 100. This looks like "
          f"a pre-v14b (102-MSA-trained) export. The chart below will still run, but its "
          f"values will NOT reflect the 100-MSA-trained model -- re-run against a true v14b "
          f"export first.")
del _check

# ══════════════════════════════════════════════════════════════════
# 1. LOAD, RENAME, & FILTER
# ══════════════════════════════════════════════════════════════════
df = pd.read_csv(INPUT_CSV_WEIGHTED)
df = df.rename(columns={
    'Adv_Weighted_Available_SF_Total': 'R&D_Weighted_Available_SF_Total',
    'Adv_Weighted_Counterfactual_SF': 'R&D_Weighted_Counterfactual_SF',
})

missing = set(TOP20_MSAS) - set(df["MSA_Name"].unique())
if missing:
    print(f"WARNING -- {len(missing)} MSA name(s) not found in {INPUT_CSV_WEIGHTED} "
          f"(check for naming mismatches, e.g. trailing whitespace):")
    for m in sorted(missing):
        print(f"    {m!r}")

df_sub = df[df["MSA_Name"].isin(TOP20_MSAS) & df["Year"].isin(RECENT_YEARS)].copy()

# Market_Category context (optional) -- pulled from the unweighted
# Section 11 export, since Market_Category is classified off the
# unweighted, size-corrected Structural_Gap, not the R&D-weighted gap.
try:
    df_cat = pd.read_csv(INPUT_CSV_RESULTS)
    cat_sub = df_cat[df_cat["MSA_Name"].isin(TOP20_MSAS) & df_cat["Year"].isin(RECENT_YEARS)]
    cat_by_msa = (
        cat_sub.groupby("MSA_Name")["Market_Category"]
        .agg(lambda x: x.mode().iat[0] if len(x.mode()) else np.nan)
        .rename("Market_Category_Unweighted_Context")
    )
except FileNotFoundError:
    cat_by_msa = None
    print(f"NOTE -- {INPUT_CSV_RESULTS} not found; proceeding without Market_Category context.")

# ══════════════════════════════════════════════════════════════════
# 2. AVERAGE 2021-2023 PER MSA — R&D-weighted actual/counterfactual
# ══════════════════════════════════════════════════════════════════
summary = (
    df_sub.groupby("MSA_Name")
    .agg(
        Actual_SF=("R&D_Weighted_Available_SF_Total", "mean"),
        Counterfactual_SF=("R&D_Weighted_Counterfactual_SF", "mean"),
    )
    .reset_index()
)
summary["Structural_Gap_SF"] = summary["Actual_SF"] - summary["Counterfactual_SF"]

if cat_by_msa is not None:
    summary = summary.merge(cat_by_msa, on="MSA_Name", how="left")

summary["_order"] = summary["MSA_Name"].map({m: i for i, m in enumerate(TOP20_MSAS)})
summary = summary.sort_values("_order").drop(columns="_order").reset_index(drop=True)

summary.to_csv(OUTPUT_CSV, index=False)
print(f"Saved: {OUTPUT_CSV}")
print(summary.round(1).to_string(index=False))

# ══════════════════════════════════════════════════════════════════
# 3. GROUPED BAR CHART — R&D-Weighted Actual vs. Counterfactual,
#    2021-2023 avg [R&D-WEIGHTED, SIZE-CORRECTED, 100-MSA PANEL]
# ══════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(13, 11))

y_pos = np.arange(len(summary))
bar_h = 0.36

ax.barh(y_pos + bar_h / 2, summary["Actual_SF"] / 1e6, height=bar_h,
        color=BLUE, alpha=0.95, label="R&D-Weighted Actual Available SF Total")
ax.barh(y_pos - bar_h / 2, summary["Counterfactual_SF"] / 1e6, height=bar_h,
        color=GRAY, edgecolor=GRAY_EDGE, linewidth=0.6,
        label="R&D-Weighted Counterfactual (pre-COVID structural model)")

# Gap label: darkened + bolded (was #555555, hard to read) for
# legibility against the light-gray counterfactual bars.
x_span = summary[["Actual_SF", "Counterfactual_SF"]].values.max() / 1e6
for i, row in summary.iterrows():
    gap = row["Structural_Gap_SF"]
    x_max = max(row["Actual_SF"], row["Counterfactual_SF"]) / 1e6
    ax.text(x_max + 0.03 * x_span, i, f"{gap:+,.0f} SF", va="center",
            fontsize=8.5, color=LABEL_COLOR, fontweight="bold")
ax.set_xlim(0, x_span * 1.18)  # extra headroom so gap labels never clip at the right edge

ax.set_yticks(y_pos)
ax.set_yticklabels(summary["MSA_Name"], fontsize=9)
ax.invert_yaxis()  # smallest R&D-Weighted base at top, matching source ranking
ax.set_xlabel("R&D-Weighted Available SF Total, 2021-2023 Avg (Million SF)", fontsize=11)
ax.set_title(
    "R&D Available SF Total, R&D-Weighted — Top 20 Small-Base Markets\n"
    "Actual vs. Counterfactual, 2021-2023 Avg | Smallest 2021-2023 R&D-Weighted Base | "
    "Size-corrected, 100-MSA Panel",
    fontsize=12, fontweight="bold")
ax.legend(fontsize=9, loc="upper center", bbox_to_anchor=(0.5, -0.06), ncol=2, frameon=False)
ax.grid(alpha=0.2, ls=":", axis="x")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(OUTPUT_PNG, dpi=300, bbox_inches="tight")
plt.show()
print(f"\nSaved: {OUTPUT_PNG}")

In [ ]:
"""PART OF FINAL MODEL — v14b-consistent (100-MSA panel) [R&D-WEIGHTED]
Growth vs. Base-Size Analysis — Top 20 Small-Base Markets (2021-2023)
=========================================================================
Reuses the EXACT computation from the standalone small-base script
(Section 2: anchor/recent base, CAGR, LQ trend) rather than
re-deriving it. This script just filters that same computation down
to the 20 MSAs you selected and adds two visualizations on top.

WHY THIS SCRIPT'S OUTPUT IS PROVABLY IDENTICAL UNDER v14b, UNLIKE
OTHER SCRIPTS IN THIS PROJECT -- worked out explicitly, not assumed:

This script's only two inputs are Adv_Weighted_Available_SF_Total and
LQ_AdvInd_Emp. Tracing each back to its source:

(Full original version history retained in git log / docs/methodology.md.)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams.update({"font.family": "serif", "font.size": 11})

BLUE = "#1a3a5c"
GRAY = "#888888"
RED = "#d62728"
GREEN = "#2ca02c"
TEAL = "#17becf"

BYMSA_CSV   = "AvailSFTotal_AdvWeighted_Actual_vs_Counterfactual_ByMSA.csv"
RESULTS_CSV = "AvailSFTotal_Counterfactual_Results.csv"

OUTPUT_SCATTER_PNG = "top20_growth_vs_base_scatter_v14b.png"
OUTPUT_LQSLOPE_PNG = "top20_lq_slope_diverging_bar_v14b.png"
OUTPUT_CSV = "Top20_SmallBase_Growth_LQTrend_v14b.csv"

ANCHOR_YEARS = [2015, 2016, 2017, 2018]
RECENT_YEARS = [2021, 2022, 2023]
TREND_YEARS  = list(range(2015, 2024))

CAGR_WINSOR_LOW, CAGR_WINSOR_HIGH = -0.95, 5.00

NON_PRESENTATION_MSAS = ['Hattiesburg, MS', 'Gulfport-Biloxi, MS']
INCLUDE_NON_PRESENTATION_MSAS = False

TOP20_MSAS = [
    "McAllen-Edinburg-Mission, TX",
    "Urban Honolulu, HI",
    "Palm Bay-Melbourne-Titusville, FL",
    "Deltona-Daytona Beach-Ormond Beach, FL",
    "Baton Rouge, LA",
    "Fresno, CA",
    "Cape Coral-Fort Myers, FL",
    "Albuquerque, NM",
    "North Port-Bradenton-Sarasota, FL",
    "Spokane-Spokane Valley, WA",
    "Knoxville, TN",
    "Chattanooga, TN-GA",
    "Albany-Schenectady-Troy, NY",
    "Ogden, UT",
    "Kiryas Joel-Poughkeepsie-Newburgh, NY",
    "Colorado Springs, CO",
    "Little Rock-North Little Rock-Conway, AR",
    "New Orleans-Metairie, LA",
    "Omaha, NE-IA",
    "Winston-Salem, NC",
]

# ══════════════════════════════════════════════════════════════════
# 0. SANITY CHECK — confirm required columns are present AND that
#    this is a 100-MSA (v14b) export, not a 102-MSA (v14 or earlier) one.
# ══════════════════════════════════════════════════════════════════
_bymsa_check = pd.read_csv(BYMSA_CSV)
_results_check = pd.read_csv(RESULTS_CSV)

_required_bymsa = {'MSA_Name', 'Year', 'Available_SF_Total', 'Adv_Weight_LQ',
                    'Adv_Weighted_Available_SF_Total'}
_required_results = {'MSA_Name', 'Year', 'LQ_AdvInd_Emp'}

_missing_bymsa = _required_bymsa - set(_bymsa_check.columns)
_missing_results = _required_results - set(_results_check.columns)
if _missing_bymsa or _missing_results:
    raise SystemExit(
        f"STOPPING -- expected columns missing. "
        f"Missing from {BYMSA_CSV}: {_missing_bymsa or 'none'}. "
        f"Missing from {RESULTS_CSV}: {_missing_results or 'none'}."
    )

_n_msas_bymsa = _bymsa_check['MSA_Name'].nunique()
_n_msas_results = _results_check['MSA_Name'].nunique()
if _n_msas_bymsa > 100 or _n_msas_results > 100:
    print(f"WARNING -- {_n_msas_bymsa} MSAs in {BYMSA_CSV}, {_n_msas_results} in {RESULTS_CSV}. "
          f"Expected 100 for a v14b run -- this looks like v14 (102-MSA-trained) output. "
          f"Per this script's own analysis, CAGR/LQ_Slope results SHOULD be identical either "
          f"way, but confirm your source files before treating this as a genuine v14b check.")

_recon = (_bymsa_check['Available_SF_Total'] * _bymsa_check['Adv_Weight_LQ']
          - _bymsa_check['Adv_Weighted_Available_SF_Total']).abs().max()
print(f"Sanity check -- Adv_Weighted_Available_SF_Total reconciles with its "
      f"components: max abs discrepancy = {_recon:.10f} "
      f"({'OK' if _recon < 1e-6 else 'MISMATCH -- investigate before trusting this analysis'})")
del _bymsa_check, _results_check, _recon

# ══════════════════════════════════════════════════════════════════
# 1. LOAD, RENAME, & MERGE
# ══════════════════════════════════════════════════════════════════
weighted = pd.read_csv(BYMSA_CSV)[['MSA_Name', 'Year', 'Adv_Weighted_Available_SF_Total']]
weighted = weighted.rename(columns={'Adv_Weighted_Available_SF_Total': 'R&D_Weighted_Available_SF_Total'})
lq = pd.read_csv(RESULTS_CSV)[['MSA_Name', 'Year', 'LQ_AdvInd_Emp']]
df = weighted.merge(lq, on=['MSA_Name', 'Year'], how='inner')

if not INCLUDE_NON_PRESENTATION_MSAS:
    df = df[~df['MSA_Name'].isin(NON_PRESENTATION_MSAS)]

# ══════════════════════════════════════════════════════════════════
# 2. SHARED BUILDING BLOCKS
# ══════════════════════════════════════════════════════════════════
anchor = (df[df['Year'].isin(ANCHOR_YEARS)]
          .groupby('MSA_Name')['R&D_Weighted_Available_SF_Total']
          .mean().rename('Anchor_Base_2015_2018'))
recent = (df[df['Year'].isin(RECENT_YEARS)]
          .groupby('MSA_Name')['R&D_Weighted_Available_SF_Total']
          .mean().rename('Recent_Level_2021_2023'))

base_growth = pd.concat([anchor, recent], axis=1).reset_index()

_floor = base_growth['Anchor_Base_2015_2018'].quantile(0.05)
base_growth['Anchor_Base_Floored'] = base_growth['Anchor_Base_2015_2018'].clip(lower=_floor)

n_years = (np.mean(RECENT_YEARS) - np.mean(ANCHOR_YEARS))
base_growth['CAGR'] = (
    (base_growth['Recent_Level_2021_2023'] / base_growth['Anchor_Base_Floored'])
    .clip(lower=1e-6) ** (1 / n_years) - 1
).clip(CAGR_WINSOR_LOW, CAGR_WINSOR_HIGH)
# NOTE (v14b): provably identical to v13/v14 -- see module docstring.

def lq_trend(group):
    g = group[group['Year'].isin(TREND_YEARS)].dropna(subset=['LQ_AdvInd_Emp'])
    if len(g) < 3:
        return pd.Series({'LQ_Slope': np.nan, 'LQ_Anchor_Mean': np.nan,
                           'LQ_Recent_Mean': np.nan})
    slope = np.polyfit(g['Year'], g['LQ_AdvInd_Emp'], 1)[0]
    lq_anchor = g[g['Year'].isin(ANCHOR_YEARS)]['LQ_AdvInd_Emp'].mean()
    lq_recent = g[g['Year'].isin(RECENT_YEARS)]['LQ_AdvInd_Emp'].mean()
    return pd.Series({'LQ_Slope': slope, 'LQ_Anchor_Mean': lq_anchor,
                       'LQ_Recent_Mean': lq_recent})
# NOTE (v14b): also provably identical to v14 -- LQ_AdvInd_Emp does not
# depend on training-panel composition (see module docstring). This is
# DIFFERENT from the v13-to-v14 transition, where the LQ denominator
# itself changed (in-panel average -> true national) and DID shift
# LQ_Slope; v14-to-v14b changes only which MSAs are in the panel, not
# how LQ_AdvInd_Emp is computed for the MSAs that remain.

lq_trends = df.groupby('MSA_Name').apply(lq_trend).reset_index()

combined = base_growth.merge(lq_trends, on='MSA_Name', how='left')

growth_cutoff_full_panel = combined['CAGR'].quantile(0.75)
# NOTE (v14b): unaffected -- CAGR is unaffected and the underlying
# 100-MSA presentation set is the same set either way.

# ══════════════════════════════════════════════════════════════════
# 3. FILTER TO THE 20 MSAs
# ══════════════════════════════════════════════════════════════════
missing = set(TOP20_MSAS) - set(combined['MSA_Name'])
if missing:
    print(f"WARNING -- not found in source CSVs: {sorted(missing)}")

top20 = combined[combined['MSA_Name'].isin(TOP20_MSAS)].copy()
top20['_order'] = top20['MSA_Name'].map({m: i for i, m in enumerate(TOP20_MSAS)})
top20 = top20.sort_values('_order').drop(columns='_order').reset_index(drop=True)

top20['Fast_Growth_vs_FullPanel'] = top20['CAGR'] >= growth_cutoff_full_panel
top20['Rising_Concentration'] = top20['LQ_Slope'] > 0
top20['Nascent_Candidate'] = top20['Fast_Growth_vs_FullPanel'] & top20['Rising_Concentration']
# NOTE (v14b): all three flags provably unaffected -- see module docstring.

top20.to_csv(OUTPUT_CSV, index=False)
print(f"Saved: {OUTPUT_CSV}")
print(f"\nFull-panel top-quartile CAGR cutoff (reference line): {growth_cutoff_full_panel:+.1%}")
print(top20[['MSA_Name', 'Anchor_Base_2015_2018', 'Recent_Level_2021_2023',
             'CAGR', 'LQ_Slope']].round(4).to_string(index=False))

# ══════════════════════════════════════════════════════════════════
# 4. SCATTER — Recent base size (x) vs CAGR (y), colored by LQ trend
# ══════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(12, 9))

colors = np.where(top20['LQ_Slope'] > 0, TEAL, RED)
sizes = 90

ax.scatter(top20['Recent_Level_2021_2023'] / 1e3, top20['CAGR'] * 100,
           c=colors, s=sizes, alpha=0.85, edgecolors='white', linewidths=0.8, zorder=3)

for _, row in top20.iterrows():
    ax.annotate(row['MSA_Name'].split(',')[0], xy=(row['Recent_Level_2021_2023'] / 1e3, row['CAGR'] * 100),
                xytext=(5, 4), textcoords='offset points', fontsize=7.5)

ax.axhline(growth_cutoff_full_panel * 100, color=BLUE, lw=1.2, ls='--', alpha=0.6,
           label=f'Full-panel top-quartile CAGR ({growth_cutoff_full_panel:+.1%})')
ax.axhline(0, color='black', lw=0.8, ls='-', alpha=0.4)

ax.set_xlabel("2021-2023 Avg R&D-Weighted Available SF (Thousand SF)", fontsize=11)
ax.set_ylabel("CAGR, 2015-18 anchor \u2192 2021-23 recent (%)", fontsize=11)
ax.set_title(
    "Growth Rate vs. Current Base Size — Top 20 Small-Base Markets [100-MSA PANEL]\n"
    "Color = rising (teal) vs. falling (red) R&D employment concentration (LQ_Slope)",
    fontsize=12, fontweight='bold')

from matplotlib.lines import Line2D
legend_elems = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=TEAL, markersize=9, label='Rising LQ_AdvInd_Emp concentration'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=RED, markersize=9, label='Falling LQ_AdvInd_Emp concentration'),
    Line2D([0], [0], color=BLUE, lw=1.2, ls='--', label=f'Full-panel top-quartile CAGR ({growth_cutoff_full_panel:+.1%})'),
]
ax.legend(handles=legend_elems, fontsize=8.5, loc='upper right')
ax.grid(alpha=0.25, ls=':')
plt.tight_layout()
plt.savefig(OUTPUT_SCATTER_PNG, dpi=300, bbox_inches='tight')
plt.show()
print(f"\nSaved: {OUTPUT_SCATTER_PNG}")

# ══════════════════════════════════════════════════════════════════
# 5. DIVERGING BAR — LQ_Slope per MSA, sorted
# ══════════════════════════════════════════════════════════════════
fig2, ax2 = plt.subplots(figsize=(11, 9))

top20_sorted = top20.sort_values('LQ_Slope')
bar_colors = [TEAL if v > 0 else RED for v in top20_sorted['LQ_Slope']]

ax2.barh(top20_sorted['MSA_Name'], top20_sorted['LQ_Slope'], color=bar_colors, alpha=0.85)
ax2.axvline(0, color='black', lw=1, ls='--')
ax2.set_xlabel("LQ_AdvInd_Emp Slope, 2015-2023 (linear trend, per year)", fontsize=11)
ax2.set_title(
    "R&D Employment Concentration Trend — Top 20 Small-Base Markets [100-MSA PANEL]\n"
    "Positive = R&D employment share concentrating faster than national average",
    fontsize=12, fontweight='bold')
ax2.tick_params(axis='y', labelsize=8.5)
ax2.grid(alpha=0.25, ls=':', axis='x')
plt.tight_layout()
plt.savefig(OUTPUT_LQSLOPE_PNG, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_LQSLOPE_PNG}")

print(f"\n{'='*60}")
print(f"NASCENT CANDIDATES within this 20 (fast growth AND rising LQ,")
print(f"using full-panel top-quartile CAGR cutoff as the bar):")
print(f"{'='*60}")
nascent_here = top20[top20['Nascent_Candidate']]
if len(nascent_here) == 0:
    print("None of the 20 clear both bars simultaneously.")
else:
    print(nascent_here[['MSA_Name', 'CAGR', 'LQ_Slope']].round(4).to_string(index=False))